# Cell-Akamai companion 01 · Inference stack: per-call agent traces
**Local Ollama on this host — NOT an Akamai GPU run, NOT vLLM.** Original workshop notebook `01_inference_stack/01_inference_stack.ipynb` is unchanged; this notebook consumes the same *question* on a different backend. Results are never pooled with Akamai runs. Status: **NOT EXECUTED** until the executed copy exists.

Traces every model call made by the four arm schedules (B0 1 call, B1 3 calls, B2/B3 3 calls) on 4 fixed ECA items. **Trace and cost only — answers are not scored here.** Token counts come from the server's `usage`; absent usage stays null.

In [ ]:
import sys, json, pathlib
sys.path[:0] = ["lane_b/companion", "lane_b/harness"]   # run from the repo root
import cakm, pandas as pd
OUT = pathlib.Path("runs/companion"); OUT.mkdir(parents=True, exist_ok=True)
MODELS = ["hf.co/LiquidAI/LFM2.5-350M-GGUF:Q4_K_M", "hf.co/LiquidAI/LFM2.5-1.2B-Instruct-GGUF:Q4_K_M"]  # small models; edit to taste
HOST = cakm.host_info(); print(json.dumps(HOST, indent=1))

In [ ]:
from run_lane_b import run_item, make_chat, ARMS
from items import make_items
items = make_items(70000, 1)  # 4 items, one per rule, disjoint from calibration seeds
rows = []
for model in MODELS:
    cfg = {"base_url": "http://127.0.0.1:11434/v1", "model": model, "temperature": 0, "decode_seed": 20261012,
           "max_tokens_per_call": 256, "item_token_cap": 3072, "item_call_cap": 3}
    base_chat = make_chat(cfg); log = []
    def chat(msgs, log=log):
        r = base_chat(msgs); log.append(r); return r
    for arm in ARMS:
        for it in items:
            n0 = len(log)
            rec = run_item(arm, it, chat, cfg)
            for k, (text, usage, dt) in enumerate(log[n0:]):
                rows.append({"model": model, "arm": arm, "item_id": it["item_id"], "call_index": k, "wall_s": dt,
                             "completion_tokens": usage, "tokens_per_s": (usage / dt) if usage and dt else None,
                             "status": "OBSERVED" if usage is not None else "OBSERVED_USAGE_UNKNOWN"})
len(rows)

In [ ]:
rec = cakm.write_receipt(OUT/"01", "traces", rows, {"module": "01", "host": HOST, "models": MODELS, "scored": False})
df = pd.DataFrame(rows)
summary = df.groupby(["model", "arm"]).agg(calls=("call_index", "size"), wall_s=("wall_s", "sum"),
                                           completion_tokens=("completion_tokens", lambda s: s.sum() if s.notna().all() else None)).reset_index()
summary["tokens_per_s_aggregate"] = summary.completion_tokens / summary.wall_s
print(rec["jsonl_sha256"]); summary

**Reading it:** the table shows what extra orchestration costs in calls, wall time and tokens on a small local model. It says nothing about correctness (see the pilot) or Akamai GPU behavior.